***Получение прогноза для тестовых данных***

Распакуем наши обработчики

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mamoUtils as mu
np.__version__, pd.__version__

('2.5.5', '3.1.1')

In [57]:
import joblib

pipeline = joblib.load('models/best_model.pkl')

bestForest_1 = pipeline['model_1']
bestForest_2 = pipeline['model_2']
bestForest_3 = pipeline['model_3']
yearsScaler = pipeline['years_scaler']
monthEncoder = pipeline['month_encoder']
unitEncoder = pipeline['unit_encoder']
skuEncoder = pipeline['sku_encoder']
scaler_1 = pipeline['param_scaler_1']
scaler_2 = pipeline['param_scaler_2']
scaler_3 = pipeline['param_scaler_3']

Далее загружаем из файла данные для прогноза

In [58]:
testDataFrame = pd.read_csv('data/test.csv')
testDataFrame.head()

,id,sku,unit,month
0,0_2021-12,0,ШТ,2021-12
1,0_2022-01,0,ШТ,2022-01
2,0_2022-02,0,ШТ,2022-02
3,1_2021-12,1,ШТ,2021-12
4,1_2022-01,1,ШТ,2022-01


In [59]:
fillDataFrame = pd.read_csv('data/train.csv')
fillDataFrame.head()

,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,379.0,250.0,317.0,172.0,193.0,173.0,121.0,318.0,328.0,208.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,68.0,65.0,72.0,82.0,85.0,69.0,75.0,100.0,72.0,89.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,334.0,380.0,409.0,309.0,388.0,438.0,404.0,293.0,274.0,381.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0,2187.0,1672.0,1665.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,222.0,258.0,289.0,662.0,759.0,616.0,365.0,321.0,392.0,409.0


In [60]:
splitter = mu.SplitDateOnColumn()
splittedData = splitter.transform(fillDataFrame)
dateSplitter = mu.SplitDateOnYearsMonth()
transformedFillData = dateSplitter.transform(splittedData)
transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue'] = 0
print(len(transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue']))
transformedFillData.head(40)

0


,sku,unit,resultValue,years,month
0,0,ШТ,128.0,2019,1
1,0,ШТ,125.0,2019,2
2,0,ШТ,179.0,2019,3
3,0,ШТ,199.0,2019,4
4,0,ШТ,204.0,2019,5
5,0,ШТ,246.0,2019,6
6,0,ШТ,172.0,2019,7
7,0,ШТ,235.0,2019,8
8,0,ШТ,238.0,2019,9
9,0,ШТ,228.0,2019,10


In [61]:
testDataFrameDates = dateSplitter.transform(testDataFrame)
testDataFrameDates.head()

,id,sku,unit,years,month
0,0_2021-12,0,ШТ,2021,12
1,0_2022-01,0,ШТ,2022,1
2,0_2022-02,0,ШТ,2022,2
3,1_2021-12,1,ШТ,2021,12
4,1_2022-01,1,ШТ,2022,1


In [62]:
ids = testDataFrameDates[['id']]
testDataFrameDates = testDataFrameDates.drop('id', axis = 1)
OneMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2021) & (testDataFrameDates['month'] == 12)]
TwoMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 1)]
ThreeMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 2)]

In [63]:
def convert_result_value(dataFrame, diff):
    df = dataFrame.copy()
    df['resultValue'] = df.groupby('sku')['resultValue'].diff(diff)
    df = df.dropna()
    return df
def get_lag_for_date(dateFrame, lagsFrom, lags):
    df = dateFrame.copy()
    columns = []
    for i in lags:
        lag_string = f'lag_{i}'
        columns += [lag_string]
        targetYear = dateFrame.iloc[0]['years']
        targetMonth = dateFrame.iloc[0]['month'] - i;
        while(targetMonth <= 0):
            targetYear -= 1
            targetMonth += 12
        lagVal = lagsFrom.loc[(lagsFrom['years'] == targetYear) & (lagsFrom['month'] == targetMonth), ['sku','resultValue']]
        lagVal = lagVal.rename(columns={'resultValue': lag_string})
        df = df.merge(lagVal, on='sku', how='left')
    return (df, columns)
def get_rolling_mean_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rm_str = f'rolling_mean_{i}'
        columns += [rm_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        mean = resValForMean.groupby('sku')[['resultValue']].mean()
        mean = mean.rename(columns={'resultValue': rm_str})
        df = df.merge(mean, on='sku', how='left')
    return (df, columns)
def get_rolling_std_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rs_str = f'rolling_std_{i}'
        columns += [rs_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        std = resValForMean.groupby('sku')[['resultValue']].std()
        std = std.rename(columns={'resultValue': rs_str})
        df = df.merge(std, on='sku', how='left')
    return (df, columns)
def get_season_ratio(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_ratio'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] / (df[season_str] + 1);
    return (df, [season_str])
def get_season_deviation(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_deviation'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] - df[season_str];
    return (df, [season_str])
def get_dataFrame_with_params(dataFrame, fillDataFrame, lags, yearsFromWindows, monthFromWindows, windows):
    baseIndexes = dataFrame.index
    
    df = dataFrame.copy()
    fillDataFrame = convert_result_value(fillDataFrame, lags[0])
    df, lag_columns = get_lag_for_date(df, fillDataFrame, lags)
    df, mean_columns = get_rolling_mean_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, std_columns = get_rolling_std_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, season_dev_columns = get_season_deviation(df, fillDataFrame, lag_columns[0])
    df.index = baseIndexes
    return (df, lag_columns + mean_columns + std_columns + season_dev_columns, [])

data_1, columns_1, freeColumn_1 = get_dataFrame_with_params(OneMonthDataFrame, transformedFillData, [1, 2, 3, 6, 12], 2021, 11, [3, 4, 6, 12])
data_1.head()

,sku,unit,years,month,lag_1,lag_2,lag_3,lag_6,lag_12,rolling_mean_3,rolling_mean_4,rolling_mean_6,rolling_mean_12,rolling_std_3,rolling_std_4,rolling_std_6,rolling_std_12,season_deviation
0,0,ШТ,2021,12,-120.0,10.0,197.0,21.0,96.0,29.0,8.75,6.0,2.75,159.351812,136.267812,106.431198,101.876150,-173.5
3,1,ШТ,2021,12,17.0,-28.0,25.0,3.0,-5.0,4.666667,5.0,1.166667,0.333333,28.571548,23.338095,19.954114,18.366636,24.0
6,2,ШТ,2021,12,107.0,-19.0,-111.0,79.0,-23.0,-7.666667,-14.25,12.0,2.583333,109.441004,90.323031,81.442004,65.692754,126.5
9,3,ШТ,2021,12,-7.0,-515.0,366.0,785.0,-17.0,-52.0,-134.75,-70.166667,1.916667,442.220533,397.193802,561.692591,526.724779,-579.0
12,4,ШТ,2021,12,17.0,71.0,-44.0,97.0,207.0,14.666667,-51.75,-42.166667,1.416667,57.535496,140.895647,133.758613,197.074263,-144.0


In [64]:
def scale_data(dataFrame, scaledColumns, freeColumns, yearsScaler, paramScaler, monthEncoder, unitEncoder, skuEncoder):
    encoded_data = encode_data(dataFrame, unitEncoder, skuEncoder)
    scaledDate = pd.DataFrame(yearsScaler.transform(dataFrame[['years']]), index = dataFrame.index, columns = ['years'])
    encodedMonth = monthEncoder.transform(dataFrame[['month']]).toarray()
    encodedMonthDF = pd.DataFrame(encodedMonth, index = dataFrame.index, columns = monthEncoder.get_feature_names_out(input_features=dataFrame[['month']].columns))
    scaledParam = pd.DataFrame(
        paramScaler.transform(
            encoded_data[['sku']].join(
                dataFrame[scaledColumns]
            )
        ), index = dataFrame.index,
        columns = ['sku'] + scaledColumns)
    resultDataFrame = scaledParam.join(encoded_data.drop('sku', axis=1)).join(encodedMonthDF).join(scaledDate).join(dataFrame[freeColumns])
    return resultDataFrame

def encode_data(dataFrame, unitEncoder, skuEncoder):
    encodedSku = pd.DataFrame(skuEncoder.transform(dataFrame[['sku']].astype(str)), index = dataFrame.index, columns = ['sku'])
    encodedUnit = unitEncoder.transform(dataFrame[['unit']]).toarray()
    encodedUnitDF = pd.DataFrame(encodedUnit, index = dataFrame.index, columns = unitEncoder.get_feature_names_out(input_features=dataFrame[['unit']].columns))
    return encodedSku.join(encodedUnitDF)

scaled_X_test_1 = scale_data(data_1, columns_1, freeColumn_1, yearsScaler, scaler_1, monthEncoder, unitEncoder, skuEncoder)

In [65]:
scaled_X_test_1.head()

,sku,lag_1,lag_2,lag_3,lag_6,lag_12,rolling_mean_3,rolling_mean_4,rolling_mean_6,rolling_mean_12,...,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,years
0,-0.200241,-0.952381,0.083333,1.683761,0.184211,0.817391,0.390909,0.132530,0.105172,-0.016667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,-0.293848,0.134921,-0.233333,0.213675,0.026316,-0.060870,0.059091,0.072289,0.005172,-0.177778,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
6,-0.063209,0.849206,-0.158333,-0.948718,0.692982,-0.217391,-0.109091,-0.236948,0.229310,-0.027778,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
9,-0.288540,-0.055556,-4.291667,3.128205,6.885965,-0.165217,-0.713636,-2.172691,-1.470690,-0.072222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
12,0.298191,0.134921,0.591667,-0.376068,0.850877,1.782609,0.195455,-0.839357,-0.891379,-0.105556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [66]:
predict_1 = pd.Series(bestForest_1.predict(scaled_X_test_1), index = scaled_X_test_1.index)

In [73]:
def transform_predict(predict, baseDataFrame, years, month):
    
    filtDF = baseDataFrame.loc[(baseDataFrame['years'] == years) & (baseDataFrame['month'] == month), 'resultValue']
    filtDF.index = predict.index
    res = predict - filtDF
    return res
transPredict_1 = transform_predict(predict_1, transformedFillData, 2021, 11)

Получили первый прогноз, самое время получать и второй и третий

In [74]:
data_2, columns_2, freeColumn_2 = get_dataFrame_with_params(TwoMonthDataFrame, transformedFillData, [2, 3, 5, 6, 12], 2021, 11, [3, 6, 12])
scaled_X_test_2 = scale_data(data_2, columns_2, freeColumn_2, yearsScaler, scaler_2, monthEncoder, unitEncoder, skuEncoder)
predict_2 = pd.Series(bestForest_2.predict(scaled_X_test_2), index = scaled_X_test_2.index)
transPredict_2 = transform_predict(predict_2, transformedFillData, 2021, 11)

In [75]:
data_3, columns_3, freeColumn_3 = get_dataFrame_with_params(ThreeMonthDataFrame, transformedFillData, [3, 4, 5, 12], 2021, 11, [3, 6, 12])
scaled_X_test_3 = scale_data(data_3, columns_3, freeColumn_3, yearsScaler, scaler_3, monthEncoder, unitEncoder, skuEncoder)
predict_3 = pd.Series(bestForest_3.predict(scaled_X_test_3), index = scaled_X_test_3.index)
transPredict_3 = transform_predict(predict_3, transformedFillData, 2021, 11)

In [76]:
allPrediction = pd.concat([transPredict_1, transPredict_2, transPredict_3]).sort_index()
allPrediction

0       -191.598015
1       -208.119830
2       -170.834490
3        -75.390170
4        -97.790460
           ...     
4963   -1253.117174
4964   -1171.521529
4965    -116.390170
4966    -137.977902
4967    -120.122473
Length: 4968, dtype: float64

In [71]:
result = ids.copy()
result['prediction'] = allPrediction.round(9)
result['prediction'] = result['prediction']
result

,id,prediction
0,0_2021-12,224.401985
1,0_2022-01,207.880170
2,0_2022-02,245.165510
3,1_2021-12,102.609830
4,1_2022-01,80.209540
...,...,...
4963,1654_2022-01,1088.882826
4964,1654_2022-02,1170.478471
4965,1655_2021-12,143.609830
4966,1655_2022-01,122.022098


In [72]:
result.to_csv('data/predict.csv', index=False)